<a href="https://colab.research.google.com/github/mehanshbarthwal-lab/search-ranking-ml/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mehanshbarthwal-lab/search-ranking-ml/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Lane: freestyle**, building on the Refresh / Content Opportunity Scoring lane.

**Project title: "Answered Away: Detecting and Costing Click-Suppression in Declining Content"**

I'm going freestyle rather than picking one of the four predefined lanes because the question I
want to ask is narrower than any single lane covers. Instead of treating "declining" as one
problem, I want to split it into two behaviorally distinct patterns: pages losing traffic because
fewer people see them at all (normal decay), versus pages that keep showing up in search just as
often but stop getting clicked (a pattern consistent with something answering the query before
the click happens, which I'm calling "answered_away"). This builds directly on the Refresh /
Content Opportunity lane's mechanics, but adds a diagnostic split and an economics layer neither
the predefined lanes nor the freestyle "AI Referral Opportunity" direction attempt, since that
direction is explicitly EDA-only due to how sparse ai_sessions data is (only ~6% of pages have
any AI-referred sessions at all in the starter CSV). My approach avoids that sparsity problem
entirely by using impressions and clicks, which every page has, as the behavioral proxy instead.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. The question: decision, action, cost of a wrong call

**Core question**: when a page is losing clicks, is it losing them because fewer people see it at
all (normal decay — impressions AND clicks both fall), or because roughly the same number of
people see it but stop clicking (a pattern consistent with something answering the query before
the click happens — impressions flat/up, clicks fall meaningfully)? I'm calling this second
pattern "answered_away" — explicitly a behavioral pattern label, not a confirmed AI-Overview
label, since no column in this dataset confirms what actually caused it.

**Unit of analysis**: one content page (content_id).

**Decision this improves**: which declining pages should a content editor fix, and with which
kind of fix — a routine content refresh for normal_decay pages, versus a structural fix (FAQ
blocks, a clearer direct-answer section near the top) for answered_away pages.

**Who acts, and how**: a content editor pulls the ranked, labeled list and picks the right fix
type per page, rather than applying the same generic "refresh" to everything flagged as declining.

**Cost of a wrong call**: applying a structural fix to a page that just needed routine freshening
wastes editor effort on the wrong intervention. Missing a real answered_away page and treating it
like normal decay means it keeps losing clicks even after a refresh that was never going to fix
the actual problem, since the page's visibility was never the issue.

**Why this earns ML over a hand-written rule**: a simple if-statement can flag this pattern after
it's already happened in the data. The harder, more valuable version is predicting which pages
are at elevated risk of falling into the answered_away pattern before it fully shows up, using
several prior-period signals together (search intent, position, competition, content type,
length) that interact in ways too tangled to hand-write as a single rule.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Quick look at the data (2-3 real numbers)

I'm building the label from real prev-30/last-30 day window comparisons already in the starter
dataset: impr_change_pct and click_change_pct, computed from impressions_last_30d vs
impressions_prev_30d and clicks_last_30d vs clicks_prev_30d. I filter to pages with
impressions_prev_30d >= 50 and clicks_prev_30d >= 3, so the percent change isn't just noise from
a tiny prior base. answered_away = impressions roughly flat or up (>= -5%) while clicks fall
meaningfully (<= -15%). normal_decay = both impressions and clicks fall together (both <= -15%).
Everything else falls into stable_other.

For the economics layer: organic and paid search are substitute goods for the same click, and cpc
is effectively the shadow price of that substitution — what the company would have to pay in
Google Ads to buy back the same traffic it's currently getting for free. I compute an
ad-equivalent value at risk for the answered_away group specifically, as
(clicks_prev_30d - clicks_last_30d) * cpc, summed across the group. This is explicitly a proxy
for what it would cost to replace that lost traffic via ads, not a literal revenue figure, since
this dataset has no conversion or order-value data.


In [2]:
import pandas as pd

# Load the starter CSV fresh
df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')

# Filter to pages with enough prior volume BEFORE computing pct change, to avoid divide-by-zero
volume_mask = (df['impressions_prev_30d'] >= 50) & (df['clicks_prev_30d'] >= 3)
df_filtered = df[volume_mask].copy()

# Compute percentage changes only on the filtered, safe subset
df_filtered['impr_change_pct'] = (
    (df_filtered['impressions_last_30d'] - df_filtered['impressions_prev_30d'])
    / df_filtered['impressions_prev_30d'] * 100
)
df_filtered['click_change_pct'] = (
    (df_filtered['clicks_last_30d'] - df_filtered['clicks_prev_30d'])
    / df_filtered['clicks_prev_30d'] * 100
)

# Define the pattern groups
def assign_pattern(row):
    if row['impr_change_pct'] >= -5 and row['click_change_pct'] <= -15:
        return 'answered_away'
    elif row['impr_change_pct'] <= -15 and row['click_change_pct'] <= -15:
        return 'normal_decay'
    else:
        return 'stable_other'

df_filtered['pattern_group'] = df_filtered.apply(assign_pattern, axis=1)

# Number 1: total pages in the usable base pool
print(f"Pages with enough prior volume to trust the pct change: {len(df_filtered):,} "
      f"(out of {len(df):,} total pages)")
print()

# Number 2: count per pattern group
print("Total pages in each pattern group:")
print(df_filtered['pattern_group'].value_counts())
print()

# Number 3: ad-equivalent value at risk for the answered_away group
answered_away_df = df_filtered[df_filtered['pattern_group'] == 'answered_away'].copy()
answered_away_df['ad_equivalent_value_at_risk'] = (
    (answered_away_df['clicks_prev_30d'] - answered_away_df['clicks_last_30d'])
    * answered_away_df['cpc'].fillna(0)
)
total_value_at_risk = answered_away_df['ad_equivalent_value_at_risk'].sum()

print(f"Ad-equivalent value at risk for 'answered_away' group: ${total_value_at_risk:,.2f}")
print("(This is a proxy for what it would cost to replace that lost traffic via ads, "
      "not a literal revenue loss — no conversion data exists in this dataset.)")

FileNotFoundError: [Errno 2] No such file or directory: '../../data/raw/content_refresh_anonymized.csv'

## 4. Careful words: what I can and can't claim

- **Can claim:** a pattern consistent with click suppression, an association between page profile and pattern type, a proxy dollar value.
- **Can never claim:** that AI Overviews specifically caused anything, real revenue loss, causal proof — no controlled experiment is possible on historical data.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
